# Project 1 – Traffic Light Object Detection

Train, validate, and run inference for the traffic-light colour detector using the
command line interface shipped with this repository. Execute each cell in order to
reproduce the full workflow on your machine.

- **Dataset root**: `Small Traffic Light.v1i.yolov11`
- **Classes**: green, off, red, wait_on, yellow
- **Key files**:
  - `Project_1_object_detection_traffic_light.py` – primary CLI entrypoint
  - `Small Traffic Light.v1i.yolov11/` – YOLO-format dataset
  - `runs/` & `outputs/` – generated training and inference artefacts

In [ ]:
from importlib import import_module, util
from pathlib import Path
import sys

def guard(module, package=None, critical=False):
    if util.find_spec(module):
        return import_module(module)
    pkg = package or module
    if module == "torch":
        print("PyTorch missing. Install CPU build via `pip install torch --index-url https://download.pytorch.org/whl/cpu`.")
        return None
    msg = f"Install via `pip install {pkg}`." if critical else f"Optional dependency `{module}` missing."
    print(msg)
    if critical:
        raise ModuleNotFoundError(module)
    return None

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

nbformat = guard("nbformat", critical=True)
torch = guard("torch")
ultralytics = guard("ultralytics")
cv2 = guard("cv2", "opencv-python")
numpy, pandas, tqdm = (guard(m) for m in ("numpy", "pandas", "tqdm"))
sahi = guard("sahi")
supervision = guard("supervision")

In [ ]:
DATA_ROOT = Path("Small Traffic Light.v1i.yolov11")
EPOCHS = 120
IMGSZ = 640
BATCH = 16
DEVICE = "auto"
SEED = 42
USE_SAHI = True
VIDEO_PATH = "inference_traffic_light_video.mp4"
WEIGHTS_PATH = "runs/best.pt"

In [ ]:
from Project_1_object_detection_traffic_light import (
    discover_dataset,
    ensure_data_yaml,
    CLASS_NAMES,
)

dataset_root = DATA_ROOT
if not dataset_root.exists():
    raise FileNotFoundError(f"Dataset root not found: {dataset_root}")
cfg = discover_dataset(dataset_root)
data_yaml = ensure_data_yaml(dataset_root)
print(f"Train images: {cfg.train_images}")
print(f"Validation images: {cfg.val_images}")
print("Classes:", ", ".join(CLASS_NAMES))
print("Data YAML:", data_yaml)

In [ ]:
from Project_1_object_detection_traffic_light import main

main([
    "train",
    "--data-root", str(DATA_ROOT),
    "--epochs", str(EPOCHS),
    "--imgsz", str(IMGSZ),
    "--batch", str(BATCH),
    "--device", DEVICE,
    "--seed", str(SEED),
    "--patience", "20",
])

In [ ]:
from Project_1_object_detection_traffic_light import main

main([
    "validate",
    "--data-root", str(DATA_ROOT),
])

In [ ]:
from Project_1_object_detection_traffic_light import main

main([
    "infer-video",
    "--weights", WEIGHTS_PATH,
    "--video", VIDEO_PATH,
    "--sahi", str(USE_SAHI).lower(),
    "--conf-thres", "0.25",
    "--iou-thres", "0.5",
])

In [ ]:
import json
from pathlib import Path

metrics_files = sorted(Path("runs").rglob("metrics.json"), key=lambda p: p.stat().st_mtime)
if metrics_files:
    latest = metrics_files[-1]
    with latest.open() as fh:
        metrics = json.load(fh)
    print("Latest metrics file:", latest)
    print("mAP50-95:", metrics.get("map50-95"))
else:
    print("No metrics.json files found. Train the model first.")
print("Annotated video:", Path("outputs/annotated.mp4"))
print("Detections CSV:", Path("outputs/detections.csv"))
print("Frame summary:", Path("outputs/summary_per_frame.csv"))

In [ ]:
if "cv2" not in globals() or cv2 is None:
    print("OpenCV not available. Install `opencv-python` to preview frames.")
else:
    import matplotlib.pyplot as plt

    video_path = Path("outputs/annotated.mp4")
    if not video_path.exists():
        print("Annotated video not found. Run the inference step first.")
    else:
        cap = cv2.VideoCapture(str(video_path))
        frames = []
        for _ in range(3):
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        cap.release()
        if not frames:
            print("No frames available for preview.")
        else:
            fig, axes = plt.subplots(1, len(frames), figsize=(15, 5))
            if len(frames) == 1:
                axes = [axes]
            for ax, frame in zip(axes, frames):
                ax.imshow(frame)
                ax.axis("off")
            plt.show()

### Notes & Next steps

- Explore larger YOLO backbones or RT-DETR for higher accuracy.
- Extend augmentation strategies (colour jitter, histogram equalisation).
- Investigate mixed-precision training once GPU acceleration is available.